# Extended Lab: Cross-Correlation, ARIMAX, and VAR — SKELETON

Practice notebook based on Chapter 12 ("Multivariate Time Series") of
*Building Statistical Models in Python*. Fill in every `# TODO` cell. A
fully worked reference is in `02_extended_lab_solutions.ipynb` — try not to
look until you've attempted each section. Read `00_theory_and_background.md`
first if any of the terms below (CCF, ARIMAX, VAR, stationarity, VIF,
Granger causality, impulse response) are unfamiliar.

**Data note:** Part 1 uses a synthetic weather-like dataset generated below
(no internet access required) that has the same qualitative structure as
the UCI Beijing weather dataset used in the book: `WSPM` (wind speed) is
driven by specific lags of `TEMP`, `PRES`, and `DEWP`, plus daily
seasonality. Part 2 uses `statsmodels`' built-in US macroeconomic dataset
(also no download required).

**Goal:** by the end, you should be able to (1) find and justify which
lagged covariates belong in an ARIMAX model using CCF, (2) fit and prune an
ARIMAX model and show it beats a no-exogenous baseline out of sample, and
(3) build, select the order of, and forecast with a VAR model — plus two
diagnostics (Granger causality, impulse response) that go beyond simple
forecasting.


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.signal import correlate
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 4)

Run this cell as-is — it builds the synthetic weather dataset you'll use for Part 1.

In [ ]:
def generate_weather_like_data(n=1500, seed=42):
    """Synthetic hourly weather data with known cross-correlation structure,
    used as a stand-in for the UCI Beijing PM2.5 weather dataset."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    temp_noise = np.zeros(n)
    for i in range(1, n):
        temp_noise[i] = 0.6 * temp_noise[i-1] + rng.normal(0, 1.0)
    TEMP = 10 + 8*np.sin(2*np.pi*t/24 - 1.2) + 0.002*t + temp_noise

    pres_noise = np.zeros(n)
    for i in range(1, n):
        pres_noise[i] = 0.7 * pres_noise[i-1] + rng.normal(0, 0.8)
    PRES = 1015 - 0.01*np.roll(TEMP, 14) + 3*np.sin(2*np.pi*t/300) + pres_noise
    PRES[:14] = 1015 + pres_noise[:14]

    dewp_noise = np.zeros(n)
    for i in range(1, n):
        dewp_noise[i] = 0.5 * dewp_noise[i-1] + rng.normal(0, 1.2)
    DEWP = -5 + 0.9*np.roll(TEMP, 2) + dewp_noise
    DEWP[:2] = -5 + dewp_noise[:2]

    wspm_noise = np.zeros(n)
    for i in range(2, n):
        wspm_noise[i] = 0.35*wspm_noise[i-1] + 0.15*wspm_noise[i-2] + rng.normal(0, 0.6)

    def lag(arr, k):
        out = np.roll(arr, k)
        out[:k] = arr[0]
        return out

    WSPM = (2.2
            + 0.05*lag(TEMP, 13) - 0.03*lag(TEMP, 24) + 0.04*lag(TEMP, 37)
            - 0.02*lag(PRES-1015, 14) + 0.015*lag(PRES-1015, 37)
            - 0.03*lag(DEWP, 2) + 0.02*lag(DEWP, 20)
            + wspm_noise)
    WSPM = np.clip(WSPM, 0.05, None)

    RAIN = np.clip(rng.exponential(0.15, n) * (rng.random(n) < 0.08), 0, None)

    return pd.DataFrame({"TEMP": TEMP, "PRES": PRES, "DEWP": DEWP, "RAIN": RAIN, "WSPM": WSPM})

df = generate_weather_like_data()
df.head()

## Part 1 — Cross-correlation & ARIMAX (wind speed forecasting)

### 1.1 Exploratory look at the target series (WSPM)

**TODO:** Plot the first 1000 points of `df.WSPM` as a line plot, and its
ACF (`sm.graphics.tsa.plot_acf`), side by side. What does the ACF tell you
about whether a plain ARIMA model would be sufficient?

In [ ]:
# TODO: plot WSPM time series and its ACF side by side
size = 1000


### 1.2 Plot the other candidate covariates (TEMP, PRES, DEWP, WSPM) as a 2x2 grid

In [ ]:
# TODO: plot TEMP, PRES, DEWP, WSPM (first 1000 points each) in a 2x2 grid of subplots


### 1.3 Write the cross-correlation function

**TODO:** Implement `plot_ccf(data_a, data_b, lag_lookback, percentile, ax, title)`
that:
1. Computes the normalized cross-correlation of `data_a` and `data_b` using
   `scipy.signal.correlate` (see the theory doc §2.1 for the formula and
   normalization).
2. Restricts the result to `+/- lag_lookback` lags.
3. Draws a stem plot on `ax`, with horizontal dashed lines at the
   significance band `z / sqrt(n)` for `z` in `{90: 1.645, 95: 1.96, 99: 2.576}`.
4. Returns `(lags, values, band)`.

Then call it for WSPM vs. PRES, TEMP, and DEWP (48-lag lookback, 95%
significance), each on its own subplot.

In [ ]:
# TODO: implement plot_ccf(...)


# TODO: call plot_ccf for WSPM vs PRES, TEMP, DEWP (3 stacked subplots)


### 1.4 Identify significant positive lags

**TODO:** Write a function that returns only the *positive* significant
lags (covariate leads the target — these are the only ones usable for
forecasting) for a given pair of series. Run it for WSPM vs. each of TEMP,
PRES, DEWP and print the top 5 by absolute correlation.

Then, looking at the CCF plots from 1.3 and this list, **manually choose a
small, spread-out set of lags** for each covariate (don't just take every
significant lag — see the theory doc §3.3 on why that causes
multicollinearity). Aim for 2–3 lags per covariate.

In [ ]:
# TODO: implement significant_positive_lags(data_a, data_b, lag_lookback, percentile=95)


# TODO: print top candidate lags for TEMP, PRES, DEWP vs WSPM


### 1.5 Build the lagged exogenous matrix

**TODO:** Using `series.shift(-lag)` for each (covariate, lag) pair you
picked in 1.4, build a DataFrame `X` of lagged exogenous columns (restricted
to the first 1000 rows), drop any resulting NaN rows, and set `y` to the
matching `WSPM` values.

In [ ]:
# TODO: build X (lagged exogenous DataFrame) and y (target), aligned and NaN-free


### 1.6 Train/test split

**TODO:** Split `X` and `y` into an 80/20 train/test split (no shuffling —
this is time series!).

In [ ]:
# TODO: X_train, X_test, y_train, y_test = ...


### 1.7 Multicollinearity check (VIF)

**TODO:** Write `vif_table(X)` using
`statsmodels.stats.outliers_influence.variance_inflation_factor` (remember
to add a constant column first) and run it on `X_train`. Which columns have
the highest VIF, and does that match your intuition from 1.3–1.4?

In [ ]:
# TODO: implement and call vif_table(X_train)


### 1.8 Fit ARIMAX with iterative p-value pruning

**TODO:** Write `fit_and_prune(y_train, X_train, order, threshold=0.05)`
that:
1. Fits `SARIMAX(y_train, exog=X_train, order=order)`.
2. Finds the exogenous coefficient with the highest p-value.
3. If it's above `threshold`, drops that column and refits; repeat.
4. Stops when all remaining exogenous p-values are below the threshold (or
   no exogenous columns remain — in which case fall back to a plain
   `SARIMAX(y_train, order=order)`).
5. Returns the final fit and the surviving exogenous columns.

Use `order=(2, 0, 0)` (matching the book's finding for the real dataset).
Print the final model summary.

In [ ]:
# TODO: implement fit_and_prune(...) and call it




### 1.9 Forecast on the held-out test set vs. a plain-ARIMA baseline

**TODO:**
1. Forecast `len(y_test)` steps ahead from your pruned ARIMAX fit, passing
   the matching exogenous test columns.
2. Fit a plain `SARIMAX(y_train, order=(2,0,0))` baseline (no exog) and
   forecast the same horizon.
3. Plot both forecasts against the actual test values, with the ARIMAX
   forecast's 95% confidence interval shaded.
4. Compute and print the MSE of each, and the percentage improvement.

In [ ]:
# TODO: forecast, plot, and compare MSE (ARIMAX vs. baseline)


### 1.10 (Extension) Rolling-origin backtest

**TODO:** Write a small walk-forward evaluation: split the (X, y) series
into several folds along time, refit on each growing training window, and
report the average out-of-sample MSE. This gives a more robust error
estimate than a single train/test split. (A sketch: pick `n_splits` cut
points, train up to each cut point, forecast the next `test_len` points,
collect the MSEs.)

In [ ]:
# TODO: implement and run a rolling-origin backtest, report mean MSE across folds


## Part 2 — VAR modeling (US macroeconomic data)

Uses `statsmodels`' built-in dataset — no download required.

### 2.1 Load and prepare the data

**TODO:** Load `sm.datasets.macrodata.load_pandas().data`, sort by
`year`/`quarter`, replace the `quarter` column with a simple running integer
index (1..n), and set it as the DataFrame index (matching the book's
approach). Display the first few rows of `realcons`, `realinv`, `realdpi`.

In [ ]:
# TODO: load and prepare the macrodata dataset


### 2.2 Step 1 — visual inspection

**TODO:** Line-plot `realcons`, `realinv`, `realdpi` together. What does the
shape suggest about stationarity?

In [ ]:
# TODO: line plot of realcons, realinv, realdpi


### 2.3 Step 2 — stationarity tests and differencing

**TODO:** Run the augmented Dickey-Fuller test (`adfuller`) on each of the
three series. Then first-difference all three (`np.diff(..., n=1)`) into a
new DataFrame `data_1d`, and re-run ADF to confirm they're now stationary.

In [ ]:
# TODO: ADF test on levels


In [ ]:
# TODO: build data_1d (first differences) and re-run ADF


### 2.4 ACF / PACF of the differenced series

**TODO:** Plot ACF and PACF (use `method="ywm"`) for each of the three
differenced series, 2x3 grid (ACF on top row, PACF on bottom). Does the PACF
suggest different AR orders for different variables? What does that tell you
about picking a single VAR order for the whole system?

In [ ]:
# TODO: ACF/PACF grid for the three differenced series


### 2.5 Step 3 — cross-correlation between candidate inputs and `realinv`

**TODO:** Reuse your `plot_ccf` function from Part 1 to plot the CCF of
`realinv` against `realcons`, and `realinv` against `realdpi`, on the
differenced data. Which variable looks like a leading indicator, and at
what lag?

In [ ]:
# TODO: CCF realinv vs realcons, realinv vs realdpi


### 2.6 Step 4 — grid search over VAR(p, q) order by AIC

**TODO:** Loop `p` from 1 to 3 and `q` from 0 to 1, fit
`VARMAX(data_1d, order=(p,q), trend="c")` for each combination (wrap in
try/except since some combinations may fail to converge), record the AIC,
and pick the best order.

In [ ]:
# TODO: grid search over (p, q), report AIC for each, pick the best


### 2.7 Fit the selected VAR model

**TODO:** Fit `VARMAX` with your best order on `data_1d` reindexed so
`realinv` is the first column (`data_1d.reindex(columns=["realinv", "realcons", "realdpi"])`).
Print the summary. Which lagged terms are significant (p < 0.05) in the
`realinv` equation?

In [ ]:
# TODO: fit the selected VAR model and print its summary


### 2.8 (Extension) Granger causality

**TODO:** Use `grangercausalitytests` to test whether `realcons`
Granger-causes `realinv`, and separately whether `realdpi` does, using
`maxlag=4`. Interpret the p-values: what do they tell you about the
"leading indicator" claim from your CCF plot in 2.5?

In [ ]:
# TODO: Granger causality tests


### 2.9 Step 5 — test forecast on a held-out tail

**TODO:** Use `var_fit.get_prediction(start=195, end=200).summary_frame(alpha=0.05)`
to get an in-sample/near-end forecast, plot it against the actual `realinv`
values with a shaded 95% CI, and display a side-by-side comparison table.

In [ ]:
# TODO: test forecast for realinv, plot + comparison table


### 2.10 Step 6 — forecast beyond the observed sample

**TODO:** Forecast 7 steps beyond the end of `data_1d` using
`get_prediction(start=len(data_1d), end=len(data_1d)+6)`, and plot history +
forecast on one chart.

In [ ]:
# TODO: out-of-sample forecast, h=7, plot


### 2.11 (Extension) Impulse response function

**TODO:** Compute `var_fit.impulse_responses(steps=10, orthogonalized=True)`
and plot the response of all three variables to a shock in the first-ordered
variable. In your own words, what is this plot showing that a point forecast
doesn't?

In [ ]:
# TODO: impulse response function, plot


## Part 3 — Wrap-up

**TODO:** Build a small summary table comparing: ARIMAX test MSE, baseline
ARIMA test MSE, and the VAR model's AIC. Then answer the discussion
questions below in a markdown cell (a few sentences each is fine).

### Discussion questions

1. The wind-speed CCF plots show an oscillating pattern rather than a
   single sharp spike. What does that tell you about the physical process
   generating the data, and why does that make "pick the single best lag" a
   worse strategy than for a one-off leading indicator?
2. The VIF table flagged the multiple temperature lags as collinear, and
   p-value pruning removed some of them one at a time. Can you think of a
   scenario where that greedy removal could throw away a *pair* of jointly
   useful lags that are each individually insignificant?
3. The book's original VAR forecast for `realinv` gets much less accurate
   around the 2007–2009 Great Recession. Why would a linear,
   constant-coefficient VAR model systematically struggle there, and what's
   one class of model built to handle regime changes like this?
4. Suppose you only had 3 significant CCF lags between wind speed and
   temperature, but needed a 10-step-ahead forecast. What are two different
   ways to extend your ARIMAX forecast horizon beyond your smallest usable
   lag, and what does each cost you in terms of compounding uncertainty?
5. If `realcons` and `realinv` were found to be cointegrated rather than
   just correlated, what would be wrong with the differencing + VAR approach
   used here, and what model would you reach for instead?

In [ ]:
# TODO: summary comparison table
